In [153]:
pip install nltk

In [154]:
pip install niapy

Import library yang dibutuhkan

In [155]:
from sklearn.model_selection import train_test_split, cross_val_score

from niapy.problems import Problem
from niapy.task import Task
from niapy.algorithms.basic import ParticleSwarmOptimization

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split

Import data

In [156]:
data = pd.read_excel('data.xlsx')
data

,userName,score,at,content
0,SENA PAMELIA,5,2023-11-04 22:53:38,sering error
1,Sahabat Berbagi'94,1,2023-11-04 19:06:57,Mao buka rekening aja ga bisa bisa
2,Nyimas Fajarwati Ns,5,2023-11-04 15:58:30,Good
3,Max Hean,1,2023-11-04 13:37:07,"Sering kali gak bisa transaksi karena ""Permint..."
4,Agus wisudayana,1,2023-11-04 12:56:47,Saya kecewa dengan bsi ini karena saya setor t...
...,...,...,...,...
2495,Wening Andayani,5,2023-08-04 14:29:20,Aplikasi yg sangat membantu dalam bertransaksi...
2496,A_ Eve Jennifer Kumarurung,5,2023-08-04 14:22:12,Mantab BSI!! Aplikasinya sangat mempermudah sa...
2497,Salma Rofiqoh,5,2023-08-04 12:57:32,Pakai Aplikasi BSI transaksi jadi lebih cepat ...
2498,Inggrid Dwi,5,2023-08-04 12:33:44,Bagus banget aplikasinya sangat membatu apalag...


Data checking

In [157]:
#getting an over-view about all columns present in the dataset
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   userName  2500 non-null   object        
 1   score     2500 non-null   int64         
 2   at        2500 non-null   datetime64[ns]
 3   content   2500 non-null   object        
dtypes: datetime64[ns](1), int64(1), object(2)
memory usage: 78.2+ KB


In [158]:
#Total number of data records that have a null value in them
data.isnull().sum()

userName    0
score       0
at          0
content     0
dtype: int64

In [159]:
count=0
for i in data['content']:
    if count==5:
        break
    print(i)
    print()
    count+=1

sering error 

Mao buka rekening aja ga bisa bisa

Good

Sering kali gak bisa transaksi karena "Permintaan kehabisan waktu", padahal koneksi internet saya bagus banget, dan ini bukan sekali aja terjadi. Hal tersebut sering terjadi ketika sore hari jam 15-18, malam hari, hari sabtu dan minggu. Nasabah BSI banyak yang gunakan bsi mobile, dan kebutuhan nasabah akan sumber keuangannya itu 24 jam. Kalau lagi urgent yang ada mau pecahin hp karena gak bisa transaksi.

Saya kecewa dengan bsi ini karena saya setor tunai saldo nya tidak masuk keatm saya, saya melakukan pengaduan sama sekali tidak membantu sama sekali buat apa diadakan cs kalo tidak membantu!!!!



Pre-processing data

In [160]:
data = data.drop(columns=['userName', 'at'], axis=1)
data

,score,content
0,5,sering error
1,1,Mao buka rekening aja ga bisa bisa
2,5,Good
3,1,"Sering kali gak bisa transaksi karena ""Permint..."
4,1,Saya kecewa dengan bsi ini karena saya setor t...
...,...,...
2495,5,Aplikasi yg sangat membantu dalam bertransaksi...
2496,5,Mantab BSI!! Aplikasinya sangat mempermudah sa...
2497,5,Pakai Aplikasi BSI transaksi jadi lebih cepat ...
2498,5,Bagus banget aplikasinya sangat membatu apalag...


In [161]:
!pip3 install contractions

In [162]:
import contractions
from tqdm import tqdm
#tqdm package is used to track the progress of work. It displays the percentage of loop done.

In [163]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
#donwloadin the stopwords of indonesia language
stopwords=stopwords.words('indonesian')
#Removing stopwords "tidak"
print('tidak' in stopwords)
stopwords.remove('tidak')
print('tidak' in stopwords)

True
False


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [164]:
import regex as re
processed_reviews=[]
for i in tqdm(data['content']):
    #Regular expression that removes all the html tags pressent in the reviews
    i=re.sub('(<[\w\s]*/?>)',"",i)
    #Expanding all the contractions present in the review to is respective actual form
    i=contractions.fix(i)
    #Removing all the special charactesrs from the review text
    i=re.sub('[^a-zA-Z0-9\s]+',"",i)
    #Removing all the digits present in the review text
    i=re.sub('\d+',"",i)
    #Making all the review text to be of lower case as well as remvoing the stopwords and words of length less than 3
    processed_reviews.append(" ".join([j.lower() for j in i.split() if j not in stopwords and len(j)>=3]))

100%|██████████| 2500/2500 [00:01<00:00, 1288.27it/s]


In [165]:
# Membuat dataframe dari data
dataframe = pd.DataFrame(processed_reviews)

# Menyimpan dataframe ke file Excel
dataframe.to_excel('data cleaned.xlsx', index=False)

In [166]:
##Pelabelan
kalimat = pd.read_excel("data cleaned.xlsx")

#skoring
# Membuka file txt
positive_words = open('positive-words.txt', 'r')
positif = positive_words.readlines()
# Menghilangkan karakter newline
positif = [word.strip() for word in positif]

negative_words = open('negative-words.txt', 'r')
negatif = negative_words.readlines()
negatif = [word.strip() for word in negatif]

In [167]:
import re
from string import punctuation
from collections import defaultdict

scores = []
for i in range(0,(len(kalimat[0]))):
  kalimat2 = str(kalimat[0][i])
  kata2 = kalimat2.split()
  positif_matches = [kata for kata in kata2 if kata in positif]
  negatif_matches = [kata for kata in kata2 if kata in negatif]
  score = len(positif_matches) - len(negatif_matches)
  scores.append(score)
  scores_df = {'score': scores, 'text': kalimat[0]}

In [168]:
#Creating a new datafram using the Processed Reviews
processed_df=pd.DataFrame(scores_df)

In [169]:
processed_df.head()

,score,text
0,0,error
1,0,mao buka rekening aja
2,0,good
3,0,sering kali gak transaksi permintaan kehabisan...
4,-3,saya kecewa bsi setor tunai saldo nya tidak ma...


In [170]:
# CONVERT SCORE TO SENTIMENT
processed_df['score'] = np.where(processed_df['score'] < 0, 0, 1)

In [171]:
processed_df = processed_df.dropna()

In [172]:
#Splitting the data into dependent and independent variables i.e, features and the target columns
X=processed_df['text']
Y=processed_df['score']

Splitting data

In [173]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, stratify=Y, random_state=1234)

In [174]:
def print_shape(a,b):
    """
    Function that prints the shape of the numpy arrays passed as arguments
    """
    print("Size of Training Samples")
    print("="*30)
    print(a.shape)
    print("Size of Testing Samples")
    print("="*30)
    print(b.shape)
print_shape(X_train,X_test)

Size of Training Samples
(1717,)
Size of Testing Samples
(737,)


Vektorisasi data review

In [175]:
from sklearn.feature_extraction.text import CountVectorizer
#Using CountVectorizer to convert text into tokens/features
vect = CountVectorizer()
#Using training data to transform text into counts of features for each message
vect.fit(X_train)
X_train_dtm = vect.transform(X_train)
X_test_dtm = vect.transform(X_test)

KNN

In [176]:
pip install pyswarm

In [177]:
from pyswarm import pso
knn = KNeighborsClassifier()
knn.fit(X_train_dtm,y_train)
print('Accuracy:', knn.score(X_test_dtm, y_test))

Accuracy: 0.8032564450474898


KNN with PSO Optimization

In [ ]:
# Optimalisasi dengan PSO
import random
random.seed(123)
def optimize_knn(k):
  knn.neighbors = int(k[0])
  return knn.score(X_train_dtm,y_train)

lb = [1]
ub = [15]

xopt,fopt = pso(optimize_knn,lb,ub)
xopt # Ini adalah nilai K paling optimal

In [ ]:
fopt # Ini adalah nilai score paling optimal

In [ ]:
xopt[0]

In [ ]:
knn_pso = KNeighborsClassifier(n_neighbors=round(xopt[0]))

knn_pso.fit(X_train_dtm, y_train)
print('Accuracy:', knn_pso.score(X_test_dtm, y_test))

Model Comparison

In [ ]:
from sklearn.metrics import roc_curve, auc
train_fpr_knn,train_tpr_knn,thresholds_knn=roc_curve(y_train,knn.predict_proba(X_train_dtm)[:,1])
test_fpr_knn,test_tpr_knn,thresholds_knn=roc_curve(y_test,knn.predict_proba(X_test_dtm)[:,1])

In [ ]:
train_fpr_knn_pso,train_tpr_knn_pso,thresholds_knn_pso=roc_curve(y_train,knn_pso.predict_proba(X_train_dtm)[:,1])
test_fpr_knn_pso,test_tpr_knn_pso,thresholds_knn_pso=roc_curve(y_test,knn_pso.predict_proba(X_test_dtm)[:,1])

In [ ]:
import matplotlib.pyplot as plt
# Make figure and axes
fig, axs = plt.subplots(2,2)

# Plotting in every axes
axs[0,0].plot(train_fpr_knn,train_tpr_knn,label="Training Accuracy="+str(round(auc(train_fpr_knn, train_tpr_knn),2)))
axs[0,0].plot(test_fpr_knn,test_tpr_knn,label="Testing Accuracy ="+str(round(auc(test_fpr_knn, test_tpr_knn),2)))
axs[0,0].legend()
axs[0,0].xlabel("Thresholds")
axs[0,0].ylabel("ACCURACY")
axs[0,0].title("KNN Training and Testing ROC Curves")

axs[1,0].plot(train_fpr_knn_pso,train_tpr_knn_pso,label="Training Accuracy="+str(round(auc(train_fpr_knn_pso, train_tpr_knn_pso),2)))
axs[1,0].plot(test_fpr_knn_pso,test_tpr_knn_pso,label="Testing Accuracy ="+str(round(auc(test_fpr_knn_pso, test_tpr_knn_pso),2)))
axs[1,0].legend()
axs[1,0].xlabel("Thresholds")
axs[1,0].ylabel("ACCURACY")
axs[1,0].title("KNN with PSO Training and Testing ROC Curves")

# Hide xlabels and tick labels
for ax in axs.flat:
  ax.label_outer()

plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
sns.heatmap(confusion_matrix(y_train,knn.predict(X_test_dtm)),annot=True)

In [ ]:
sns.heatmap(confusion_matrix(y_test,knn_pso.predict(X_test_dtm)),annot=True)